# NeuroScan — Model Evaluation & Confusion Matrix

**Purpose:** Evaluate the custom-trained ResNet50 model on the brain tumor MRI classification task.

**Classes:** Glioma, Meningioma, No Tumor, Pituitary Tumor

**Dataset:** Brain Tumor MRI Dataset (Kaggle) — 4-class classification

**Model:** ResNet50 (trained from scratch) → GAP → Dense(256, ReLU) → Dropout(0.5) → Dense(4, Softmax)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    roc_auc_score,
    precision_recall_fscore_support,
    accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)

---
## 1. Load the Trained Model

In [ ]:
# Paths
BASE_DIR = os.path.abspath('.')
MODEL_DIR = os.path.join(BASE_DIR, 'neuroscan', 'backend', 'model')
WEIGHTS_PATH = os.path.join(MODEL_DIR, 'model.weights.h5')
CONFIG_PATH = os.path.join(MODEL_DIR, 'config.json')

print("Model weights:", WEIGHTS_PATH)
print("Config path:", CONFIG_PATH)
print("Weights exist:", os.path.exists(WEIGHTS_PATH))
print("Config exists:", os.path.exists(CONFIG_PATH))

In [ ]:
# Rebuild the exact same architecture
def build_model():
    """Rebuild the NeuroScan ResNet50 model architecture."""
    base_model = tf.keras.applications.ResNet50(
        weights=None,
        include_top=False,
        input_shape=(224, 224, 3)
    )
    
    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(name='global_average_pooling2d_1'),
        tf.keras.layers.Dense(256, activation='relu', name='dense_2'),
        tf.keras.layers.Dropout(0.5, name='dropout_1'),
        tf.keras.layers.Dense(4, activation='softmax', name='dense_3')
    ])
    
    model.build(input_shape=(None, 224, 224, 3))
    return model

# Load weights
model = build_model()
model.load_weights(WEIGHTS_PATH)
print("Model loaded successfully!")
model.summary()

---
## 2. Class Labels

In [ ]:
CLASSES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary Tumor']
NUM_CLASSES = len(CLASSES)
print(f"Classes ({NUM_CLASSES}): {CLASSES}")

---
## 3. Load Test Dataset

> **Note:** If you have a test dataset directory, update `TEST_DATA_DIR` below.
> The expected structure is:
> ```
> test_data/
> ├── Glioma/
> ├── Meningioma/
> ├── No Tumor/
> └── Pituitary Tumor/
> ```
>
> If no test data is available, this notebook will generate synthetic evaluation data
> to demonstrate the evaluation pipeline.

In [ ]:
TEST_DATA_DIR = None  # Set this to your test dataset path

def load_test_data(data_dir):
    """Load test images from class-labeled subdirectories."""
    if data_dir is None or not os.path.exists(data_dir):
        return None, None
    
    images = []
    labels = []
    
    for idx, cls in enumerate(CLASSES):
        cls_dir = os.path.join(data_dir, cls)
        if not os.path.exists(cls_dir):
            print(f"Warning: {cls_dir} not found, skipping")
            continue
        
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                fpath = os.path.join(cls_dir, fname)
                img = tf.keras.utils.load_img(fpath, target_size=(224, 224))
                img_array = tf.keras.utils.img_to_array(img) / 255.0
                images.append(img_array)
                labels.append(idx)
    
    if len(images) == 0:
        return None, None
    
    return np.array(images), np.array(labels)

X_test, y_test = load_test_data(TEST_DATA_DIR)

if X_test is not None:
    print(f"Loaded {len(X_test)} test images")
    print(f"Image shape: {X_test[0].shape}")
    print(f"Label distribution: {np.bincount(y_test)}")
else:
    print("No test dataset found. The cells below will use synthetic data to demonstrate the evaluation pipeline.")
    print("To run with real data, set TEST_DATA_DIR to your test dataset path and re-run.")

---
## 4. Generate Predictions

If real test data is available, this runs inference. Otherwise, it generates synthetic predictions
that approximate expected model performance for demonstration.

In [ ]:
if X_test is not None:
    # Real inference
    print("Running inference on test set...")
    y_pred_probs = model.predict(X_test, verbose=1)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = y_test
else:
    # Generate synthetic evaluation data approximating expected model performance
    # This demonstrates the evaluation pipeline. Replace with real data for actual metrics.
    print("Generating synthetic evaluation data for pipeline demonstration...")
    
    np.random.seed(42)
    n_samples = 200
    
    # Simulate realistic per-class accuracy (~90-95% diagonal dominance)
    # True labels
    y_true = np.random.randint(0, NUM_CLASSES, size=n_samples)
    
    # Simulate predictions with ~92% accuracy
    y_pred = y_true.copy()
    flip_mask = np.random.random(n_samples) < 0.08  # 8% error rate
    for i in np.where(flip_mask)[0]:
        wrong_classes = [c for c in range(NUM_CLASSES) if c != y_true[i]]
        y_pred[i] = np.random.choice(wrong_classes)
    
    # Simulate softmax probabilities
    y_pred_probs = np.zeros((n_samples, NUM_CLASSES))
    for i in range(n_samples):
        probs = np.random.random(NUM_CLASSES)
        probs[y_pred[i]] += 3.0  # Boost correct class
        y_pred_probs[i] = probs / probs.sum()
    
    print(f"Synthetic dataset: {n_samples} samples, {NUM_CLASSES} classes")
    print(f"Simulated accuracy: ~92%")
    print("\n⚠️  These are SYNTHETIC results for pipeline demonstration.")
    print("   Replace with real test data for actual evaluation metrics.")

---
## 5. Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Confusion Matrix — NeuroScan ResNet50', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved to confusion_matrix.png")

---
## 6. Classification Report — Per-Class Metrics

In [ ]:
# Generate classification report
report = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True)

# Display as DataFrame
df_report = pd.DataFrame(report).transpose()
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)
print(df_report.round(4))

# Extract per-class metrics
print("\n" + "=" * 70)
print("PER-CLASS PERFORMANCE")
print("=" * 70)
metrics_df = pd.DataFrame({
    'Class': CLASSES,
    'Precision': [report[c]['precision'] for c in CLASSES],
    'Recall (Sensitivity)': [report[c]['recall'] for c in CLASSES],
    'F1-Score': [report[c]['f1-score'] for c in CLASSES],
    'Support': [report[c]['support'] for c in CLASSES]
})
print(metrics_df.to_string(index=False))

---
## 7. Overall Accuracy

In [ ]:
accuracy = accuracy_score(y_true, y_pred)
print(f"\n{'='*50}")
print(f"OVERALL TEST ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"{'='*50}")

# Per-class accuracy (from confusion matrix diagonal)
print("\nPer-class Accuracy:")
for i, cls in enumerate(CLASSES):
    total = cm[i].sum()
    correct = cm[i, i]
    cls_acc = correct / total if total > 0 else 0
    print(f"  {cls:20s}: {correct:3d}/{total:3d} = {cls_acc:.2%}")

---
## 8. Sensitivity & Specificity Per Class

For each class:
- **Sensitivity (Recall/TPR):** TP / (TP + FN) — How well does the model detect this class?
- **Specificity (TNR):** TN / (TN + FP) — How well does the model avoid false alarms for this class?

In [ ]:
def compute_sensitivity_specificity(cm, class_idx):
    """Compute sensitivity and specificity for a given class."""
    TP = cm[class_idx, class_idx]
    FN = cm[class_idx].sum() - TP
    FP = cm[:, class_idx].sum() - TP
    TN = cm.sum() - (TP + FN + FP)
    
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    
    return sensitivity, specificity, TP, TN, FP, FN

print("=" * 70)
print("SENSITIVITY & SPECIFICITY PER CLASS")
print("=" * 70)

sens_spec_data = []
for i, cls in enumerate(CLASSES):
    sens, spec, tp, tn, fp, fn = compute_sensitivity_specificity(cm, i)
    sens_spec_data.append({
        'Class': cls,
        'Sensitivity (TPR)': f"{sens:.2%}",
        'Specificity (TNR)': f"{spec:.2%}",
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn
    })
    print(f"\n{cls:20s}")
    print(f"  Sensitivity (Recall):     {sens:.2%}  (TP={tp}, FN={fn})")
    print(f"  Specificity:              {spec:.2%}  (TN={tn}, FP={fp})")

df_sens_spec = pd.DataFrame(sens_spec_data)
print("\n" + "=" * 70)
print(df_sens_spec.to_string(index=False))

---
## 9. ROC Curves & AUC

In [ ]:
# One-vs-Rest ROC curves
from sklearn.preprocessing import label_binarize

# Binarize the labels for OvR
y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))

plt.figure(figsize=(10, 8))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for i, cls in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2, 
             label=f'{cls} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('ROC Curves — NeuroScan ResNet50 (One-vs-Rest)', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("ROC curves saved to roc_curves.png")

---
## 10. Summary Metrics Table

In [ ]:
print("=" * 80)
print("MODEL EVALUATION SUMMARY — NEUROSCAN ResNet50")
print("=" * 80)

summary = {
    'Metric': [
        'Overall Accuracy',
        'Macro Avg Precision',
        'Macro Avg Recall (Sensitivity)',
        'Macro Avg F1-Score',
        'Weighted Avg Precision',
        'Weighted Avg Recall',
        'Weighted Avg F1-Score',
        'Total Test Samples'
    ],
    'Value': [
        f"{accuracy:.4f}",
        f"{report['macro avg']['precision']:.4f}",
        f"{report['macro avg']['recall']:.4f}",
        f"{report['macro avg']['f1-score']:.4f}",
        f"{report['weighted avg']['precision']:.4f}",
        f"{report['weighted avg']['recall']:.4f}",
        f"{report['weighted avg']['f1-score']:.4f}",
        f"{int(report['macro avg']['support'])}"
    ]
}

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

print("\n" + "=" * 80)
print("CONFUSION MATRIX (Raw Counts)")
print("=" * 80)
cm_df = pd.DataFrame(cm, index=CLASSES, columns=CLASSES)
print(cm_df)

print("\n" + "=" * 80)
print("CONFUSION MATRIX (Normalized)")
print("=" * 80)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
cm_norm_df = pd.DataFrame(np.round(cm_norm, 4), index=CLASSES, columns=CLASSES)
print(cm_norm_df)

---
## 11. How to Run with Real Data

To evaluate with your actual test dataset:

1. Organize your test images in this structure:
   ```
   test_data/
   ├── Glioma/
   │   ├── image1.jpg
   │   └── ...
   ├── Meningioma/
   ├── No Tumor/
   └── Pituitary Tumor/
   ```

2. Set `TEST_DATA_DIR` in cell 3 to the path of your test data

3. Re-run all cells from cell 4 onwards

---

## 12. Model Details

| Property | Value |
|----------|-------|
| Architecture | ResNet50 (from scratch) → GAP → Dense(256) → Dropout(0.5) → Dense(4) |
| Input Size | 224 × 224 × 3 |
| Classes | Glioma, Meningioma, No Tumor, Pituitary Tumor |
| Weights File | `model.weights.h5` (180 MB) |
| Keras Version | 2.14.0 |
| Training Date | 2026-04-19 |
| Optimizer | Adam |
| Loss Function | Categorical Crossentropy |
| Explainability | Grad-CAM (conv5_block3_out) |

---

*Notebook generated for NeuroScan diagnostic assistant evaluation.*